In [ ]:
import sys
sys.path.append("..")

from src.models import RandomFourierFeatures

import torch
from torch.optim import Adam
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Generating Data

In [ ]:
torch.manual_seed(42)

resolution = 512

coords = torch.linspace(-1, 1, resolution)

cosines = 12
low_frequencies = torch.rand(cosines-3) * 3*torch.pi
high_frequencies = torch.rand(3) * 20*torch.pi
frequencies = torch.hstack([low_frequencies, high_frequencies])
#frequencies = torch.rand(cosines) * 10*torch.pi
offsets = torch.rand(cosines) * 2*torch.pi

coss = torch.vstack([torch.cos(coords * freq + offset) for freq, offset in zip(frequencies, offsets)])
combined = torch.sum(coss, dim=0)

test_mask = torch.arange(resolution) % 2 == 0
test_coords = coords[test_mask]
test_values = combined[test_mask]
remaining_coords = coords[~test_mask]
remaining_values = combined[~test_mask]

sparsities = [0.5, 0.05]
sparse_coords = []
sparse_values = []
for sparsity in sparsities:
    sparse_mask = torch.bernoulli(torch.ones(remaining_coords.numel()) * sparsity).to(torch.bool)
    sparse_coords.append(remaining_coords[sparse_mask])
    sparse_values.append(remaining_values[sparse_mask])

width = 6.27
height = width * 1/3
fig, axes = plt.subplots(1, 3, figsize=(6.27, height), constrained_layout=True)
for ax in axes:
    ax.plot(coords, combined)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].plot(test_coords, test_values, '.')
axes[0].set_title("Test samples")
for i in range(len(sparsities)):
    axes[i+1].plot(sparse_coords[i], sparse_values[i], '.')
    axes[i+1].set_title(f"sparsity = {sparsities[i]}")

#plt.savefig("figures/1d-example-data.pgf")

In [ ]:
inputs = coords.view(-1, 1)
targets = combined.view(-1, 1)
test_inputs = test_coords.view(-1, 1)
test_targets = test_values.view(-1, 1)

# Fitting on all data

In [ ]:
model = RandomFourierFeatures(1, 128, 128, 3, 1, scale=8.0)
optimizer = Adam(params=model.parameters())

epochs = 100
train_losses = torch.zeros(epochs)

for i in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = F.mse_loss(outputs, targets)
    train_losses[i] = loss.item()
    loss.backward()
    optimizer.step()

plt.plot(train_losses)

In [ ]:
outputs = model(inputs).detach()


plt.plot(coords, combined)
plt.plot(coords, outputs.view(resolution))

# Fitting on sparse samples

In [ ]:
scales = [0.125, 0.5, 4., 16.]
models = [[] for _ in sparsities]
for j in range(len(sparsities)):
    for scale in scales:
        model = RandomFourierFeatures(1, 128, 128, 3, 1, scale=scale)
        optimizer = Adam(params=model.parameters())
        
        epochs = 100
        train_losses = torch.zeros(epochs)
        test_losses = torch.zeros(epochs)

        inputs = sparse_coords[j].view(-1, 1)
        targets = sparse_values[j].view(-1, 1)
        for i in range(epochs):
            model.train()
            optimizer.zero_grad()
            outputs = model(inputs)
            train_loss = F.mse_loss(outputs, targets)
            train_losses[i] = train_loss.item()
            train_loss.backward()
            optimizer.step()
            test_outputs = model(test_inputs)
            test_losses[i] = F.mse_loss(test_outputs, test_targets).item()
    
        models[j].append(model)
        plt.figure()
        plt.plot(train_losses)
        plt.plot(test_losses)

In [ ]:
inputs = coords.view(-1, 1)

width = 5.27
height = width * (len(scales)*4) / (len(sparsities)*6)
fig, axes = plt.subplots(len(scales), len(sparsities), figsize=(width, height), constrained_layout=True)
for i in range(len(scales)):
    for j in range(len(sparsities)):
        outputs = models[j][i](inputs).detach()
        test_outputs = models[j][i](test_inputs).detach()
        test_rmse = torch.sqrt(F.mse_loss(test_outputs, test_targets))
    
        axes[i, j].plot(coords, combined)
        axes[i, j].plot(coords, outputs.view(resolution), color="red")
        axes[i, j].plot(sparse_coords[j], sparse_values[j], '.')
        axes[i, j].text(-0.85, 4.5, f"\\footnotesize{{test RMSE = {test_rmse:.2f}}}")
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        if j == 0:
            axes[i, j].set_ylabel(f"scale = {scales[i]}")
        if i == 0:
            axes[i, j].set_title(f"sparsity = {sparsities[j]}")

#plt.savefig("figures/1d-example-results-alternative-pattern.pgf")

# Using guide

In [ ]:
from scipy.signal import medfilt
import numpy as np

In [ ]:
noisy = combined + torch.randn(len(combined))
filtered = torch.tensor(medfilt(noisy))
guide_targets = filtered.view(-1, 1)
plateus = np.sort(np.random.choice(512, 4, replace=False))
for i in range(0, len(plateus), 2):
    filtered[plateus[i]: plateus[i+1]] = 0
plt.plot(coords, filtered)

In [ ]:
plt.plot(coords, combined)

In [ ]:
plt.plot(coords, filtered)

In [ ]:
guided_model = RandomFourierFeatures(1, 128, 128, 3, 1, scale=4.0)
optimizer = Adam(params=guided_model.parameters())

epochs = 100
train_losses = torch.zeros(epochs)

for i in range(epochs):
    guided_model.train()
    optimizer.zero_grad()
    outputs = guided_model(inputs)
    loss = F.mse_loss(outputs, guide_targets)
    train_losses[i] = loss.item()
    loss.backward()
    optimizer.step()

plt.plot(train_losses)

In [ ]:
outputs = guided_model(inputs).detach()


plt.plot(coords, filtered)
plt.plot(coords, outputs.view(resolution))

In [ ]:
outputs = guided_model(inputs).detach()
test_outputs = guided_model(test_inputs).detach()
test_rmse = torch.sqrt(F.mse_loss(test_outputs, test_targets))

plt.plot(coords, combined)
plt.plot(coords, outputs.view(resolution), color="red")
plt.text(-0.85, 4.5, f"\\footnotesize{{test RMSE = {test_rmse:.2f}}}")

In [ ]:
state_dict = guided_model.state_dict()
models = []
for j in range(len(sparsities)):
    model = RandomFourierFeatures(1, 128, 128, 3, 1, scale=4)
    model.load_state_dict(state_dict)
    
    optimizer = Adam(params=model.parameters())
    
    epochs = 100
    train_losses = torch.zeros(epochs)
    test_losses = torch.zeros(epochs)

    test_inputs = test_coords.view(-1, 1)
    test_targets = test_values.view(-1, 1)
    inputs = sparse_coords[j].view(-1, 1)
    targets = sparse_values[j].view(-1, 1)
    for i in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(inputs)
        train_loss = F.mse_loss(outputs, targets)
        train_losses[i] = train_loss.item()
        train_loss.backward()
        optimizer.step()
        test_outputs = model(test_inputs)
        test_losses[i] = F.mse_loss(test_outputs, test_targets).item()

    models.append(model)
    plt.figure()
    plt.plot(train_losses)
    plt.plot(test_losses)

In [ ]:
inputs = coords.view(-1, 1)

width = 5.27
height = width / 2
fig, axes = plt.subplots(1, len(sparsities), figsize=(width, height), constrained_layout=True)
for j in range(len(sparsities)):
    outputs = models[j](inputs).detach()
    test_outputs = models[j](test_inputs).detach()
    test_rmse = torch.sqrt(F.mse_loss(test_outputs, test_targets))

    axes[j].plot(coords, combined)
    axes[j].plot(coords, outputs.view(resolution), color="red")
    axes[j].plot(sparse_coords[j], sparse_values[j], '.')
    axes[j].text(-0.85, 4.5, f"\\footnotesize{{test RMSE = {test_rmse:.2f}}}")
    axes[j].set_xticks([])
    axes[j].set_yticks([])
    axes[j].set_title(f"sparsity = {sparsities[j]}")

#plt.savefig("figures/1d-example-results.pgf")

# Other approaches

Simple interpolation, signal reconstruction, guided interpolation?